### Cell 1 — Setup

In [0]:
# ============================================================
#  04_ML_CLUSTERING — TOURGO ML PIPELINE
#  Day 5: Feature Engineering cho K-Means
#  Tạo ra: ml_customer_features (7 features)
# ============================================================

from pyspark.sql.functions import (
    col, count, sum as _sum, avg, countDistinct,
    when, datediff, to_date, lit,
    round as _round, coalesce
)
from pyspark.sql.types import DoubleType, IntegerType
from datetime import date

spark.sql("USE tourgo")

# Ngày export — dùng để tính days_active
# Thay bằng ngày thật khi chạy
EXPORT_DATE = str(date.today())

print("   Setup xong")
print(f"  EXPORT_DATE = {EXPORT_DATE}")
print("   Sẽ tạo: ml_customer_features")

   Setup xong
  EXPORT_DATE = 2026-06-01
   Sẽ tạo: ml_customer_features


### Cell 2 — Đọc Silver tables

In [0]:
# Đọc các Silver tables cần dùng
df_bookings = spark.read.table("silver_bookings")
df_revenues  = spark.read.table("silver_revenues")
df_reviews   = spark.read.table("silver_reviews")
df_users     = spark.read.table("silver_users") \
                    .filter(col("role") == "Customer")

print("    Silver tables loaded:")
print(f"   bookings : {df_bookings.count():,}")
print(f"   revenues : {df_revenues.count():,}")
print(f"   reviews  : {df_reviews.count():,}")
print(f"   customers: {df_users.count():,}")

    Silver tables loaded:
   bookings : 1,321
   revenues : 788
   reviews  : 271
   customers: 111


### Cell 3 — Feature 1–4: từ bookings

In [0]:
# ── Features từ bảng bookings ──────────────────────────────
# Feature 1: total_bookings   — mức độ hoạt động
# Feature 2: confirmed_bookings — tỷ lệ hoàn thành
# Feature 3: cancelled_bookings — để tính cancellation_rate
# Feature 4: unique_tours     — sự đa dạng hành vi
# Feature 5: cancellation_rate — độ tin cậy

print("  Tính features từ bookings...")

df_booking_features = df_bookings \
    .groupBy("user_id") \
    .agg(
        count("id").alias("total_bookings"),
        count(when(col("status") == "confirmed", 1))
            .alias("confirmed_bookings"),
        count(when(col("status") == "cancelled", 1))
            .alias("cancelled_bookings"),
        countDistinct("tour_id").alias("unique_tours")
    ) \
    .withColumn(
        "cancellation_rate",
        _round(
            when(col("total_bookings") > 0,
                 col("cancelled_bookings").cast(DoubleType()) /
                 col("total_bookings").cast(DoubleType())
            ).otherwise(0.0),
            3
        )
    )

print(f" Booking features: {df_booking_features.count()} users có booking")
df_booking_features.show(5, truncate=False)

  Tính features từ bookings...
 Booking features: 112 users có booking
+-------+--------------+------------------+------------------+------------+-----------------+
|user_id|total_bookings|confirmed_bookings|cancelled_bookings|unique_tours|cancellation_rate|
+-------+--------------+------------------+------------------+------------+-----------------+
|117    |16            |10                |5                 |16          |0.313            |
|76     |17            |6                 |4                 |17          |0.235            |
|123    |10            |7                 |1                 |10          |0.1              |
|98     |13            |5                 |4                 |13          |0.308            |
|51     |13            |10                |1                 |13          |0.077            |
+-------+--------------+------------------+------------------+------------+-----------------+
only showing top 5 rows


### Cell 4 — Feature: total_spent từ revenues

In [0]:
# ── Feature total_spent từ revenues ────────────────────────
# revenues.payment_id → bookings.id → user_id

print("  Tính total_spent từ revenues...")

df_spending = df_revenues \
    .join(
        df_bookings.select(
            col("id").alias("booking_id"),
            col("user_id")
        ),
        col("payment_id") == col("booking_id"),
        "left"
    ) \
    .groupBy("user_id") \
    .agg(
        _round(_sum("total_amount"), 0).alias("total_spent")
    )

print(f" Spending features: {df_spending.count()} users có chi tiêu")
df_spending.show(5, truncate=False)

  Tính total_spent từ revenues...
 Spending features: 113 users có chi tiêu
+-------+-----------+
|user_id|total_spent|
+-------+-----------+
|93     |5.75E7     |
|28     |1.0115E8   |
|33     |8.75E7     |
|30     |4.115E7    |
|29     |6.25E7     |
+-------+-----------+
only showing top 5 rows


### Cell 5 — Feature: avg_rating_given từ reviews

In [0]:
# ── Feature avg_rating_given từ reviews ────────────────────

print("  Tính avg_rating_given từ reviews...")

df_rating_features = df_reviews \
    .groupBy("user_id") \
    .agg(
        _round(avg("rating"), 2).alias("avg_rating_given"),
        count("id").alias("review_count")
    )

print(f" Rating features: {df_rating_features.count()} users đã review")
df_rating_features.show(5, truncate=False)

  Tính avg_rating_given từ reviews...
 Rating features: 106 users đã review
+-------+----------------+------------+
|user_id|avg_rating_given|review_count|
+-------+----------------+------------+
|47     |4.67            |3           |
|28     |4.0             |5           |
|75     |4.25            |4           |
|124    |4.33            |3           |
|65     |3.75            |4           |
+-------+----------------+------------+
only showing top 5 rows


### Cell 6 — Feature: days_active từ users

In [0]:
# ── Feature days_active từ users ───────────────────────────
# Tính số ngày kể từ khi tham gia → chia 30 = số tháng

print("Tính days_active từ users...")

df_user_age = df_users \
    .withColumn(
        "days_active",
        datediff(
            to_date(lit(EXPORT_DATE)),
            to_date(col("date_joined"))
        ).cast(IntegerType())
    ) \
    .select(
        col("id").alias("user_id"),
        col("days_active")
    )

print(f"User age features: {df_user_age.count()} customers")
df_user_age.show(5, truncate=False)

Tính days_active từ users...
User age features: 111 customers
+-------+-----------+
|user_id|days_active|
+-------+-----------+
|28     |3          |
|29     |322        |
|30     |344        |
|33     |421        |
|93     |-219       |
+-------+-----------+
only showing top 5 rows


### Cell 7 — JOIN tất cả features

In [0]:
# ── JOIN tất cả features thành 1 bảng ──────────────────────
print("  Joining all features...")

df_features = df_users.select(col("id").alias("user_id")) \
    .join(df_booking_features, "user_id", "left") \
    .join(df_spending,         "user_id", "left") \
    .join(df_rating_features,  "user_id", "left") \
    .join(df_user_age,         "user_id", "left") \
    .fillna({
        "total_bookings":    0,
        "confirmed_bookings":0,
        "cancelled_bookings":0,
        "unique_tours":      0,
        "cancellation_rate": 0.0,
        "total_spent":       0.0,
        "avg_rating_given":  0.0,
        "review_count":      0,
        "days_active":       0
    })

# Lưu feature table
df_features.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ml_customer_features")

total = df_features.count()
with_bookings = df_features.filter(col("total_bookings") > 0).count()

print(f"   ml_customer_features: {total:,} customers")
print(f"   Có bookings: {with_bookings:,} ({with_bookings/total*100:.1f}%)")
print(f"   Không có bookings: {total - with_bookings:,}")
df_features.show(5, truncate=False)

  Joining all features...
   ml_customer_features: 111 customers
   Có bookings: 111 (100.0%)
   Không có bookings: 0
+-------+--------------+------------------+------------------+------------+-----------------+-----------+----------------+------------+-----------+
|user_id|total_bookings|confirmed_bookings|cancelled_bookings|unique_tours|cancellation_rate|total_spent|avg_rating_given|review_count|days_active|
+-------+--------------+------------------+------------------+------------+-----------------+-----------+----------------+------------+-----------+
|28     |15            |10                |3                 |15          |0.2              |1.0115E8   |4.0             |5           |3          |
|29     |10            |7                 |2                 |10          |0.2              |6.25E7     |4.0             |2           |322        |
|30     |15            |9                 |4                 |13          |0.267            |4.115E7    |4.0             |4           |344    

### Cell 8 — Distribution Check (gửi cho [A] Hà)

In [0]:
# ============================================================
#  DISTRIBUTION CHECK — gửi kết quả này cho [A] Hà validate
# ============================================================

df_feat = spark.read.table("ml_customer_features")
total = df_feat.count()

print("=" * 55)
print(" FEATURE DISTRIBUTION CHECK — GỬI CHO [A] HÀ")
print("=" * 55)

checks = {
    "Users tổng":                total,
    "Users có booking":          df_feat.filter(col("total_bookings") > 0).count(),
    "Users booking > 5":         df_feat.filter(col("total_bookings") > 5).count(),
    "Users đã chi tiêu":         df_feat.filter(col("total_spent") > 0).count(),
    "Users đã review":           df_feat.filter(col("review_count") > 0).count(),
    "Users confirmed > 0":       df_feat.filter(col("confirmed_bookings") > 0).count(),
    "Users cancellation > 50%":  df_feat.filter(col("cancellation_rate") > 0.5).count(),
}

for label, val in checks.items():
    pct = f"({val/total*100:.1f}%)" if total > 0 else ""
    print(f"  {label:<30}: {val:>5,} {pct}")

print("=" * 55)

# Thống kê mô tả 7 features
print("\n DESCRIBE — 7 FEATURES:")
df_feat.select(
    "total_bookings", "confirmed_bookings", "total_spent",
    "cancellation_rate", "avg_rating_given", "unique_tours", "days_active"
).describe().show(truncate=False)

# Quyết định augment
pct_no_booking = (total - checks["Users có booking"]) / total * 100
print("=" * 55)
if pct_no_booking > 80:
    print(f"--ERROR--  {pct_no_booking:.1f}% users không có booking")
    print("-> CẦN AUGMENT DATA — báo [D] Khang chạy seed_fake_data.py")
    print("   Target: ít nhất 200 bookings để K-Means có ý nghĩa")
else:
    print(f"--OK-- {100 - pct_no_booking:.1f}% users có booking — đủ data")
    print("-> KHÔNG cần augment, tiến hành Day 6 (Train K-Means)")
print("=" * 55)

 FEATURE DISTRIBUTION CHECK — GỬI CHO [A] HÀ
  Users tổng                    :   111 (100.0%)
  Users có booking              :   111 (100.0%)
  Users booking > 5             :   110 (99.1%)
  Users đã chi tiêu             :   111 (100.0%)
  Users đã review               :   105 (94.6%)
  Users confirmed > 0           :   111 (100.0%)
  Users cancellation > 50%      :     2 (1.8%)

 DESCRIBE — 7 FEATURES:
+-------+------------------+------------------+--------------------+-------------------+------------------+-----------------+------------------+
|summary|total_bookings    |confirmed_bookings|total_spent         |cancellation_rate  |avg_rating_given  |unique_tours     |days_active       |
+-------+------------------+------------------+--------------------+-------------------+------------------+-----------------+------------------+
|count  |111               |111               |111                 |111                |111               |111              |111               |
|mean   |11

### Cell 9 — Preview feature table (để chụp screenshot)

In [0]:
# ============================================================
#  PREVIEW — Top 10 customers theo total_spent
#  Chụp screenshot cell này cho docs/screenshots/day5/
# ============================================================

print(" TOP 10 CUSTOMERS (by total_spent):")
spark.read.table("ml_customer_features") \
    .orderBy(col("total_spent").desc()) \
    .select(
        "user_id",
        "total_bookings",
        "confirmed_bookings",
        "total_spent",
        "cancellation_rate",
        "avg_rating_given",
        "unique_tours",
        "days_active"
    ) \
    .limit(10) \
    .show(truncate=False)

print("\n Day 5 hoàn tất!")
print("   -> Gửi output Cell 8 cho [A] Hà")
print("   -> Chờ [A] Hà xác nhận trước khi sang Day 6")

 TOP 10 CUSTOMERS (by total_spent):
+-------+--------------+------------------+-----------+-----------------+----------------+------------+-----------+
|user_id|total_bookings|confirmed_bookings|total_spent|cancellation_rate|avg_rating_given|unique_tours|days_active|
+-------+--------------+------------------+-----------+-----------------+----------------+------------+-----------+
|75     |19            |9                 |1.561E8    |0.526            |4.25            |19          |-101       |
|121    |18            |11                |1.3755E8   |0.111            |3.67            |18          |0          |
|100    |20            |10                |1.3075E8   |0.15             |4.0             |20          |215        |
|58     |19            |10                |1.26E8     |0.316            |3.75            |19          |-286       |
|90     |13            |8                 |1.255E8    |0.308            |4.0             |13          |400        |
|91     |15            |11          